# Draft Audit -- Full Demo (Layer 1 + Layer 2 + Layer 3)

This notebook runs on **your own free Google account**, using a free T4 GPU. Nothing here costs you anything, and nothing is shared with anyone else's session.

**Before running:** go to `Runtime` -> `Change runtime type` -> set Hardware accelerator to **T4 GPU** -> Save. Then run the cells below in order (Shift+Enter on each).

Layer 1 (pattern rules) is instant. Layer 2 (statistical scorer) and Layer 3 (adversarial robustness check) download about 3GB of model weights on first use and take a few minutes -- that's normal, not a hang.

See the project's [FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/FINDINGS.md), [LAYER2_FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/LAYER2_FINDINGS.md), and [LAYER3_FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/LAYER3_FINDINGS.md) for what these numbers actually mean and their real, documented limitations. Nothing in this notebook is a validated, calibrated "this is AI" verdict.

## 1. Get the code

Clones the public repo directly -- no manual uploads needed. Note: `corpus/` and `ai_corpus/` are intentionally excluded from the repo (private, consent-gated documents), so cloning does *not* give you access to any of the project's real test documents. You'll paste in your own text below.

In [ ]:
!git clone https://github.com/Gh12890/ai-writing-detector.git
%cd ai-writing-detector
!pip install -q -r requirements.txt

## 2. Paste the text you want to analyze

Edit the text between the triple quotes below, then run this cell.

In [ ]:
text = """
Paste your text here, replacing this placeholder.
""".strip()

print(f"Loaded {len(text.split())} words.")

## 3. Layer 1 -- pattern rules (instant, no GPU needed)

In [ ]:
from layer1 import PatternScorer

result = PatternScorer().analyze(text)

print(f"Word count: {result.word_count}")
print(f"Total flags: {result.flag_count}")
print(f"Structural /1000w: {result.structural_density_per_1000w}")
print(f"Lexical /1000w: {result.lexical_density_per_1000w}")
if result.suppressed_count:
    print(f"{result.suppressed_count} flag(s) suppressed as legal boilerplate.")

by_tier = result.flags_by_tier()
print(f"\nStructural flags ({len(by_tier['structural'])}):")
for f in by_tier["structural"]:
    print(f"  [{f.rule_id}] {f.match_text[:60]!r}")
print(f"\nLexical flags ({len(by_tier['lexical'])}):")
for f in by_tier["lexical"]:
    print(f"  [{f.rule_id}] {f.match_text[:60]!r}")

## 4. Layer 2 -- statistical scorer (Binoculars)

Downloads ~3GB of model weights on first run. Takes a few minutes the first time; fast after that within the same session.

In [ ]:
from layer2_binoculars import load_models, binoculars_score, binoculars_score_full_document

print("Loading models (first run downloads ~3GB)...")
observer, performer, tok, device = load_models()
print(f"Loaded. Running on: {device}")

truncated_score = binoculars_score(text, observer, performer, tok, device)
full_result = binoculars_score_full_document(text, observer, performer, tok, device)

print(f"\nTruncated score (first ~512 tokens): {truncated_score:.4f}")
if full_result["pooled_score"] is not None:
    print(f"Full-document pooled score: {full_result['pooled_score']:.4f} "
          f"({full_result['num_chunks']} chunks)")
print("\nCloser to 1.0 = more machine-like, further = more human-like, per the "
      "Binoculars paper's convention. Neither score is validated or calibrated -- "
      "there is no threshold that means 'this is AI.' See LAYER2_FINDINGS.md.")

## 5. Layer 3 -- adversarial robustness check

**Not a tool to help evade detection.** This answers one question honestly: if this text is flagged, how much does that verdict survive a cheap, automated paraphrase? Uses the same model already loaded above (Qwen2.5-1.5B-Instruct) -- no extra download.

On this project's own test data, results were inconsistent -- sometimes the paraphrase reduced flags, sometimes it increased them, and on the project's own held-out final test set the attack made both test documents *more* detectable, not less. See LAYER3_FINDINGS.md for the full, honest account. Treat any single result below as one data point, not a reliable prediction.

In [ ]:
from layer3_adversarial import chunked_paraphrase

print("Generating local paraphrase attack (chunked, ~120 words at a time)...")
paraphrase = chunked_paraphrase(text, performer, tok, device)

para_result = PatternScorer().analyze(paraphrase)
para_binoculars = binoculars_score(paraphrase, observer, performer, tok, device)

word_ratio = para_result.word_count / result.word_count if result.word_count else 0

print(f"\nOriginal:    {result.word_count} words, {result.flag_count} flags, "
      f"Binoculars {truncated_score:.4f}")
print(f"Paraphrase:  {para_result.word_count} words, {para_result.flag_count} flags, "
      f"Binoculars {para_binoculars:.4f}")
print(f"Word count ratio: {word_ratio:.2f} "
      f"(ratios well below ~0.7 mean the model compressed rather than paraphrased -- "
      f"treat the comparison as less reliable if so)")

orig_rules = sorted({f.rule_id for f in result.flags})
para_rules = sorted({f.rule_id for f in para_result.flags})
print(f"\nOriginal rules:   {orig_rules}")
print(f"Paraphrase rules: {para_rules}")

print("\n--- Paraphrased text (for reference only) ---\n")
print(paraphrase)

---
Questions about the methodology, the honest limitations, or the real measured results behind any number above: see the four findings documents in the [repo](https://github.com/Gh12890/ai-writing-detector) -- `FINDINGS.md`, `LAYER2_FINDINGS.md`, `HUMANIZER_FINDINGS.md`, `LAYER3_FINDINGS.md`.